# Part 20 — manuscript table fragments

Parts 12 to 19 write their results as CSV files. This notebook renders the `.tex`
fragments that `main.tex` and `supporting.tex` input and that no earlier notebook
produces.

Run the cells in order. The first two check the working directory before anything
is written; nothing is created until Drive is mounted and the pipeline folder is
found, so a wrong path fails loudly instead of making empty folders.

Each fragment is compared against the manuscript copy after it is written.


In [ ]:
# =====================================================================
# 1 — mount Drive and locate the pipeline
#
# Nothing is created here. If the folder is not found the cell stops, so
# that a wrong path cannot leave empty directories behind.
# =====================================================================

import os, sys

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    print("not running on Colab; set WORK by hand below")
except ValueError as e:
    # Already mounted, or the mount point holds files from an earlier cell.
    print("mount note:", e)
    print("if this says the mountpoint contains files, restart the runtime "
          "(Runtime > Restart session) and run this cell first.")

WORK = "/content/drive/MyDrive/ames_pipeline"

# The manuscript source tree, for the comparison in the last cell.
# Upload the tables/ folder from the manuscript package to Drive and point
# this at it. Set to None to skip the comparison.
REFERENCE = f"{WORK}/manuscript/tables"

EXPECTED = ["data", "results", "models", "explain", "figures", "tables"]

if not os.path.isdir("/content/drive/MyDrive"):
    sys.exit("Drive is not mounted. Run this cell again and complete the "
             "account prompt.")

if not os.path.isdir(WORK):
    print(f"NOT FOUND: {WORK}")
    print("\nTop level of this Drive:")
    for name in sorted(os.listdir("/content/drive/MyDrive"))[:40]:
        print("   ", name)
    sys.exit("Set WORK to the pipeline folder and run this cell again.")

print(f"found {WORK}\n")
for sub in sorted(os.listdir(WORK)):
    p = os.path.join(WORK, sub)
    n = len(os.listdir(p)) if os.path.isdir(p) else "file"
    mark = "" if sub in EXPECTED or sub.startswith("ckpt") else "   <-- unexpected"
    print(f"   {sub:16s} {str(n):>6}{mark}")

missing = [d for d in EXPECTED if not os.path.isdir(os.path.join(WORK, d))]
if missing:
    print(f"\nmissing subfolders: {missing}")
    print("this is probably not the folder the pipeline wrote to.")


In [ ]:
# =====================================================================
# 2 — inventory of the source files
#
# Lists which result CSVs this notebook needs, whether each is present,
# and its columns. If a table fails to build later, the columns printed
# here say why.
# =====================================================================

import pandas as pd

R = f"{WORK}/results"
T = f"{WORK}/tables"

NEEDED = {
    "table77_auroc.csv":                 ["tab_auroc"],
    "table92_auroc_pvalues.csv":         ["tab_auroc_p"],
    "table96_dod_pvalues.csv":           ["tab_dod"],
    "table97_regime_auroc.csv":          ["tab_regime_auroc"],
    "table99_dod_by_partition.csv":      ["tab_dod_replication"],
    "table101_tuning_optimism.csv":      ["tabS17_optimism"],
    "table60_architecture_summary.csv":  ["tab_architectures"],
    "table88_regimes_by_partition.csv":  ["tab_replication_regimes"],
}

if not os.path.isdir(R):
    print(f"no results folder at {R}")
else:
    have = set(os.listdir(R))
    print(f"{len(have)} files in results/\n")
    for name, used_by in NEEDED.items():
        if name in have:
            cols = list(pd.read_csv(f"{R}/{name}", nrows=1).columns)
            n = sum(1 for _ in open(f"{R}/{name}")) - 1
            print(f"OK      {name}  ({n} rows)")
            print(f"        columns: {cols}")
        else:
            print(f"MISSING {name}   needed by {', '.join(used_by)}")
    extra = sorted(f for f in have
                   if f.startswith("table") and f not in NEEDED)
    if extra:
        print(f"\nother table files present ({len(extra)}):")
        for f in extra:
            print("   ", f)


In [ ]:
# =====================================================================
# 3 — the LaTeX writer
#
# Matches the fragments in the manuscript source: table (never table*),
# [htbp], \footnotesize from five columns, \small below that.
# =====================================================================

import re, difflib
import numpy as np

os.makedirs(T, exist_ok=True)      # safe: WORK was verified in cell 1


def _esc(t):
    for a, b in [("\\", r"\textbackslash "), ("_", r"\_"), ("%", r"\%"),
                 ("&", r"\&"), ("#", r"\#")]:
        t = t.replace(a, b)
    return t


def _tex(df, caption, label, path, align=None, note=None,
         floatfmt="{:.3f}", size=None):
    cols = list(df.columns)
    align = align or ("l" + "r" * (len(cols) - 1))
    size = size or ("\\footnotesize" if len(cols) >= 5 else "\\small")

    def fmt(v):
        if isinstance(v, float):
            return "--" if pd.isna(v) else floatfmt.format(v)
        return str(v)

    lines = [r"\begin{table}[htbp]", r"\centering",
             r"\caption{" + caption + "}", r"\label{" + label + "}",
             size, r"\setlength{\tabcolsep}{4pt}",
             r"\begin{tabular}{" + align + "}", r"\toprule",
             " & ".join(str(c) for c in cols) + r" \\", r"\midrule"]
    for _, row in df.iterrows():
        lines.append(" & ".join(fmt(v) for v in row.values) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}"]
    if note:
        lines.append(r"\begin{flushleft}\footnotesize " + note +
                     r"\end{flushleft}")
    lines.append(r"\end{table}")
    open(path, "w").write("\n".join(lines) + "\n")


def holm(p):
    """Holm step-down, as Part 18 applies it."""
    p = np.asarray(p, dtype=float)
    order = np.argsort(p); k = len(p); run = 0.0; adj = np.empty(k)
    for r_, i_ in enumerate(order):
        run = max(run, (k - r_) * p[i_]); adj[i_] = min(run, 1.0)
    return adj


def holm_col(d, family=None):
    """p_holm if the stage wrote one, otherwise computed here."""
    if "p_holm" in d.columns and d.p_holm.notna().all():
        return d.p_holm.values
    sub = d if family is None else d[family]
    out = np.full(len(d), np.nan)
    out[family if family is not None else slice(None)] = holm(
        sub[col(sub, "p_raw", "p")].values)
    return out


def sig(p):
    return "$^{*}$" if (pd.notna(p) and p < 0.05) else ""


def pm(v, n=3):
    return "--" if pd.isna(v) else f"{v:+.{n}f}"


def ci(lo, hi, n=3):
    if pd.isna(lo) or pd.isna(hi):
        return "--"
    return f"{lo:+.{n}f} to {hi:+.{n}f}"


def need(name):
    p = f"{R}/{name}"
    if not os.path.exists(p):
        raise FileNotFoundError(f"{name} not in results/")
    return pd.read_csv(p)


def col(df, *names):
    """First column present under any of these names."""
    for n in names:
        if n in df.columns:
            return n
    raise KeyError(f"none of {names} in {list(df.columns)}")


BUILT, SKIPPED = [], {}


def build(fragment):
    def wrap(fn):
        try:
            fn()
            BUILT.append(fragment)
            print(f"built   {fragment}")
        except Exception as e:
            SKIPPED[fragment] = f"{type(e).__name__}: {e}"
            print(f"skipped {fragment}  ({type(e).__name__}: {e})")
        return fn
    return wrap


In [ ]:
# =====================================================================
# 4 — tab_auroc and tab_auroc_p     (source: table77_auroc.csv, Part 16)
# =====================================================================

ARM_LABEL = {
    "atom_degree": "atom degree", "degree": "atom degree",
    "random": "random ranking", "random_ranking": "random ranking",
    "gin_ig": "trained GIN, IG", "gat_ig": "trained GAT, IG",
    "gcn_ig": "trained GCN, IG",
    "gradient_x_input": "Gradient x Input", "gxi": "Gradient x Input",
    "saliency": "Saliency",
}
ARM_ORDER = ["atom degree", "random ranking", "trained GIN, IG",
             "trained GAT, IG", "trained GCN, IG", "Gradient x Input",
             "Saliency"]


def _auroc_frame():
    d = need("table77_auroc.csv").copy()
    a = col(d, "arm", "ranking", "label")
    d["label"] = d[a].map(lambda x: ARM_LABEL.get(str(x).lower(), str(x)))
    d = d[d.label.isin(ARM_ORDER)].copy()
    d["order"] = d.label.map(ARM_ORDER.index)
    return d.sort_values("order")


@build("tab_auroc.tex")
def _():
    d = _auroc_frame()
    tie = col(d, "mean_tie_fraction", "tie_fraction")
    out = pd.DataFrame({
        "Ranking": d.label,
        "$n$": d[col(d, "n", "n_models")].astype(int),
        "AUROC": d[col(d, "auroc")].map(lambda v: f"{v:.3f}"),
        "Tie fraction": d[tie].map(
            lambda v: "--" if pd.isna(v) else f"{v:.2f}"),
    })
    _tex(out,
         "Agreement as the area under the ROC curve for alert membership, "
         "computed per molecule and averaged, with the share of the top-30\\% "
         "set that a tie at the cut decides.",
         "tab:auroc", f"{T}/tab_auroc.tex", align="lccc",
         note="A value of 0.5 is chance. The tie fraction is $0.87$ for atom "
              "degree, which takes five values across the dataset and fewer "
              "within a molecule. The floor is not zero: a continuous ranking "
              "still has the atom at the cut itself, which is $1/k$ and gives "
              "$0.18$ for a random ranking, so the attributions at about "
              "$0.19$ are effectively tie-free. Differences from the degree "
              "ranking, with their intervals and corrected $p$-values, are in "
              "Table~2 of the manuscript. TreeSHAP is absent because its "
              "atom-level scores were not retained.")


@build("tab_auroc_p.tex")
def _():
    # source: table92_auroc_pvalues.csv (Part 18)
    # columns: arm, family, n, n_eval, difference, ci_low, ci_high, p_raw
    d = need("table92_auroc_pvalues.csv").copy()
    a = col(d, "arm", "ranking")
    d["label"] = d[a].map(lambda x: ARM_LABEL.get(str(x).lower(), str(x)))
    d = d[d.label.isin(ARM_ORDER) & (d.label != "atom degree")].copy()
    d["order"] = d.label.map(ARM_ORDER.index)
    d = d.sort_values("order").reset_index(drop=True)

    # Holm runs over the five attribution comparisons; the random ranking
    # is a control on the measure and carries no corrected value.
    fam = (d.label != "random ranking").values
    d["ph"] = holm_col(d, fam)

    out = pd.DataFrame({
        "Ranking": d.label,
        "$n$": d[col(d, "n", "n_models")].astype(int),
        "$\\Delta$ vs.\\ degree": [pm(v) + sig(p if f else praw)
                                  for v, p, f, praw in
                                  zip(d[col(d, "difference")], d.ph, fam,
                                      d[col(d, "p_raw", "p")])],
        "95\\% CI": [ci(x, y) for x, y in
                     zip(d[col(d, "ci_low")], d[col(d, "ci_high")])],
        "$p$": d[col(d, "p_raw", "p")].map(lambda v: f"{v:.4f}"),
        "$p$ (Holm)": [("--" if not f else f"{v:.4f}")
                       for v, f in zip(d.ph, fam)],
    })
    _tex(out,
         "Agreement on the threshold-free measure, with bootstrap $p$-values "
         "and Holm correction across the five attribution comparisons.",
         "tab:auroc_p", f"{T}/tab_auroc_p.tex", align="lccccc",
         note="An asterisk marks a comparison resolved at 0.05 after "
              "correction. The random ranking is a control on the measure "
              "rather than a claim about a model and carries no corrected "
              "value; twenty independent rankings were drawn for it, as "
              "twenty models were fitted for each architecture.")


In [ ]:
# =====================================================================
# 5 — tab_regime_auroc        (source: table97_regime_auroc.csv, Part 19)
# =====================================================================

REGIME_LABEL = {
    "full": "Full library", "full_library": "Full library",
    "no_polycyclic": "Heteroatom alerts", "heteroatom": "Heteroatom alerts",
    "polycyclic_only": "Fused aromatic only",
}
REGIME_ORDER = ["Full library", "Heteroatom alerts", "Fused aromatic only"]
RANK_LABEL = {"atom_degree": "Atom degree", "degree": "Atom degree",
              "gin": "trained GIN", "gat": "trained GAT",
              "gcn": "trained GCN"}
RANK_ORDER = ["Atom degree", "trained GIN", "trained GAT", "trained GCN"]


@build("tab_regime_auroc.tex")
def _():
    d = need("table97_regime_auroc.csv").copy()
    d["reg"] = d[col(d, "regime")].map(
        lambda r: REGIME_LABEL.get(str(r).lower(), str(r)))
    def _rank(a):
        s = str(a).lower()
        if "degree" in s:
            return "Atom degree"
        for k, v in RANK_LABEL.items():
            if k in s:
                return v
        return str(a)
    d["rank"] = d[col(d, "arm", "ranking")].map(_rank)
    d = d[d.reg.isin(REGIME_ORDER) & d["rank"].isin(RANK_ORDER)].copy()
    d["ro"] = d.reg.map(REGIME_ORDER.index)
    d["ao"] = d["rank"].map(RANK_ORDER.index)
    d = d.sort_values(["ro", "ao"])

    ne, au = col(d, "n_eval", "n"), col(d, "auroc")
    df_, lo, hi = (col(d, "difference", "diff_vs_degree"),
                   col(d, "ci_low", "diff_low"),
                   col(d, "ci_high", "diff_high"))
    p, tie = col(d, "p_raw", "p"), col(d, "tie_fraction", "mean_tie_fraction")

    # Holm runs across the six cells of the two reduced regimes; the full
    # library reproduces the published values and is a control, so it
    # carries no correction.
    fam = ((d["rank"] != "Atom degree") & (d.reg != "Full library")).values
    d = d.reset_index(drop=True)
    fam = ((d["rank"] != "Atom degree") & (d.reg != "Full library")).values
    ph = np.full(len(d), np.nan)
    ph[fam] = holm(d.loc[fam, p].values)
    d["ph"] = ph

    rows, seen = [], set()
    for _, r in d.iterrows():
        first = r.reg not in seen
        seen.add(r.reg)
        base = r["rank"] == "Atom degree"
        marked = pd.notna(r.ph) and r.ph < 0.05
        rows.append({
            "Regime": f"{r.reg} ($n$ = {int(r[ne])})" if first else "",
            "Ranking": r["rank"],
            "AUROC": f"{r[au]:.3f}",
            "$\\Delta$": "--" if base else pm(r[df_]) + ("$^{*}$" if marked else ""),
            "95\\% CI": "--" if base else ci(r[lo], r[hi]),
            "Tie": f"{r[tie]:.2f}",
        })
    _tex(pd.DataFrame(rows),
         "The regime decomposition on the threshold-free measure. The full "
         "library reproduces the published values and is a control; Holm is "
         "applied across the six cells of the two reduced regimes.",
         "tab:regime_auroc", f"{T}/tab_regime_auroc.tex", align="llcccc",
         note="An asterisk marks a difference resolved after correction. The "
              "last column is the share of the top-30\\% set that a tie at the "
              "cut decides. It is $0.97$ for the degree ranking under the "
              "fused aromatic alert against $0.82$ under the heteroatom "
              "alerts, so the lift measure was doing different work in the "
              "two regimes; the threshold-free measure does not depend on it.")


In [ ]:
# =====================================================================
# 6 — tabS17_optimism, tab_dod, tab_replication_regimes, tab_architectures
# =====================================================================

ARCHS = ["GIN", "GAT", "GCN"]


@build("tabS17_optimism.tex")
def _():
    d = need("table101_tuning_optimism.csv").copy()
    out = pd.DataFrame({
        "Model": d[col(d, "model")].map(_esc),
        "Family": d[col(d, "family")].str.capitalize(),
        "All": d[col(d, "auc_all")].map(lambda v: f"{v:.3f}"),
        "Search set": d[col(d, "auc_tuned")].map(lambda v: f"{v:.3f}"),
        "Rest": d[col(d, "auc_rest")].map(lambda v: f"{v:.3f}"),
        "Optimism": d[col(d, "optimism")].map(pm),
        "95\\% CI": [ci(a, b) for a, b in
                     zip(d[col(d, "ci_low")], d[col(d, "ci_high")])],
    })
    _tex(out,
         "Out-of-fold performance on the 725 compounds the hyperparameter "
         "search saw and on the 6,530 it did not.",
         "tab:si_optimism", f"{T}/tabS17_optimism.tex", align="llccccc",
         note="The two subsets are not matched on difficulty, so the absolute "
              "gap is not interpretable; only its spread across models is. "
              "The graph models carry -0.027 (-0.043 to -0.011) less optimism "
              "than the classical ones, so the family contrast of $+0.0146$ "
              "is if anything suppressed by the non-nested search rather than "
              "produced by it. The search set is a tenth of the data, so the "
              "effect on the pooled contrast is about a tenth of that "
              "difference; the figure here is not directly comparable with "
              "the contrast.")


@build("tab_dod.tex")
def _():
    # source: table96_dod_pvalues.csv (Part 18), one row per architecture
    d = need("table96_dod_pvalues.csv").copy()
    ac = col(d, "architecture", "arch", "model")
    d["A"] = d[ac].str.upper()
    d = d[d.A.isin(ARCHS)].copy()
    d["o"] = d.A.map(ARCHS.index)
    d = d.sort_values("o").reset_index(drop=True)
    d["ph"] = holm_col(d, np.ones(len(d), dtype=bool))
    out = pd.DataFrame({
        "Architecture": d.A,
        "Mutagenic": d[col(d, "diff_mutagenic", "mutagenic")].map(pm),
        "Non-mutagenic": d[col(d, "diff_non_mutagenic",
                               "non_mutagenic")].map(pm),
        "Difference": [pm(v) + sig(p) for v, p in zip(d[col(d, "dod")], d.ph)],
        "95\\% CI": [ci(x, y) for x, y in
                     zip(d[col(d, "ci_low")], d[col(d, "ci_high")])],
        "$p$ (Holm)": d.ph.map(lambda v: f"{v:.4f}"),
    })
    _tex(out,
         "The difference between the two evaluation sets under a signed "
         "aggregation, with Holm correction across the three architectures.",
         "tab:dod", f"{T}/tab_dod.tex", align="lccccc",
         note="Columns two and three are each architecture's advantage over "
              "the degree ranking on that set. The pre-registered direction "
              "is negative: a model using the chemistry should rank alert "
              "atoms lower where they do not carry the endpoint. Ten models "
              "per architecture, 237 mutagenic and 168 non-mutagenic "
              "molecules.")


@build("tab_dod_replication.tex")
def _():
    # source: table99_dod_by_partition.csv (Part 19), one row per partition
    d = need("table99_dod_by_partition.csv").copy()
    pc = col(d, "partition")

    def _label(v):
        s = str(v).strip().lower()
        if s in ("0", "designated", "primary"):
            return "Designated"
        return f"{s} (corrected)"

    d["P"] = d[pc].map(_label)
    d["o"] = [0 if p == "Designated" else 1 for p in d.P]
    d = d.sort_values(["o", "P"]).reset_index(drop=True)
    pv = col(d, "p_raw", "p")
    out = pd.DataFrame({
        "Partition": d.P,
        "Models": d[col(d, "n_models")].astype(int),
        "Mutagenic": d[col(d, "diff_mutagenic")].map(pm),
        "Non-mutagenic": d[col(d, "diff_non_mutagenic")].map(pm),
        "Difference": [pm(v) + sig(p)
                       for v, p in zip(d[col(d, "dod")], d[pv])],
        "95\\% CI": [ci(x, y) for x, y in
                     zip(d[col(d, "ci_low")], d[col(d, "ci_high")])],
    })
    _tex(out,
         "The specificity check repeated on the corrected partitions. "
         "Columns three and four are GIN's advantage over the degree ranking "
         "on each set.",
         "tab:dod_replication", f"{T}/tab_dod_replication.tex",
         align="lccccc",
         note="Five models on each corrected partition against ten on the "
              "designated one, so the intervals are wider. The pre-registered "
              "direction is negative. It holds on one of the three corrected "
              "partitions, is not resolved on a second, and reverses on the "
              "third.")


READING_LABEL = {
    "heteroatom": "Heteroatom alerts", "no_polycyclic": "Heteroatom alerts",
    "smarts": "Fused aromatic (SMARTS)",
    "polycyclic_only": "Fused aromatic (SMARTS)",
    "rings_2": "Fused, two or more rings",
    "rings_3": "Fused, three or more",
}
READING_ORDER = ["Heteroatom alerts", "Fused aromatic (SMARTS)",
                 "Fused, two or more rings", "Fused, three or more"]


@build("tab_replication_regimes.tex")
def _():
    # Part 17 computes this but does not cache it: its cached() helper
    # refuses a frame whose note column is non-empty. Clear that column
    # there, or re-run the stage, before this fragment can be built.
    d = need("table88_regimes_by_partition.csv").copy()
    d["row"] = d[col(d, "regime")].map(
        lambda r: READING_LABEL.get(str(r).lower(), str(r)))
    d = d[d.row.isin(READING_ORDER)].copy()
    pc = col(d, "partition")
    dc = col(d, "difference")
    pv = col(d, "p_raw", "p")
    parts = sorted(d[pc].unique())
    wide = {"Regime": READING_ORDER}
    for part in parts:
        sub = d[d[pc] == part].set_index("row")
        wide[str(part)] = [pm(sub.loc[r, dc]) + sig(sub.loc[r, pv])
                           for r in READING_ORDER]
    _tex(pd.DataFrame(wide),
         "The regime decomposition on the same three partitions. Entries are "
         "the difference between a trained GIN and the degree ranking.",
         "tab:replication_regimes", f"{T}/tab_replication_regimes.tex",
         align="lccc",
         note="Every one of the twelve intervals excludes zero, and the sign "
              "is the same on every partition and under every reading of the "
              "fused-aromatic alert. On the designated partition the two "
              "SMARTS regimes give $+0.442$ and $-0.231$.")


@build("tab_architectures.tex")
def _():
    # source: table60_architecture_summary.csv (Part 12), one row per arch
    s = need("table60_architecture_summary.csv").copy()
    sa = col(s, "architecture", "arch")
    s["A"] = s[sa].str.upper()
    s = s[s.A.isin(ARCHS)].set_index("A")
    lm, ls = col(s, "lift_mean"), col(s, "lift_sd")
    rho = col(s, "spearman_ig_degree", "rho")
    rr = col(s, "r_with_degree_overlap", "r_per_molecule", "pearson_r")
    rows = []
    for a in ARCHS:
        r = s.loc[a]
        rows.append({
            "Architecture": a,
            "Lift": f"{float(r[lm]):.3f} $\\pm$ {float(r[ls]):.3f}",
            "$\\rho$(IG, degree)": f"{float(r[rho]):.3f}",
            "$r$ per molecule": f"{float(r[rr]):.3f}",
        })
    _tex(pd.DataFrame(rows),
         "Attribution across three architectures, twenty models each. "
         "$\\rho$ is the within-molecule association between Integrated "
         "Gradients and atom degree; $r$ is the correlation between the "
         "per-molecule agreement of the architecture and of the degree "
         "ranking.",
         "tab:architectures", f"{T}/tab_architectures.tex", align="lccc",
         note="The comparison of each architecture against the baseline is "
              "given in Table~1 of the manuscript, where the uncertainty of "
              "the baseline is carried. The association with degree is "
              "weakest in GIN and strongest in GAT.")


In [ ]:
# =====================================================================
# 7 — verification
#
# Compare every fragment written above against the manuscript copy.
# Nothing in the manuscript tree is modified; it is only read.
# =====================================================================

print(f"built   : {len(BUILT)}")
print(f"skipped : {len(SKIPPED)}")
for frag, why in SKIPPED.items():
    print(f"   {frag:28s} {why}")

if REFERENCE and os.path.isdir(REFERENCE):
    same, differ, absent = [], [], []
    for frag in sorted(BUILT):
        a, b = f"{REFERENCE}/{frag}", f"{T}/{frag}"
        if not os.path.exists(a):
            absent.append(frag)
            continue
        (same if open(a).read() == open(b).read() else differ).append(frag)

    print(f"\nidentical to the manuscript : {len(same)}")
    for f in same:
        print("   ", f)
    print(f"different                   : {len(differ)}")
    for frag in differ:
        print(f"\n--- {frag}")
        d = difflib.unified_diff(
            open(f"{REFERENCE}/{frag}").read().splitlines(),
            open(f"{T}/{frag}").read().splitlines(),
            "manuscript", "rebuilt", lineterm="", n=0)
        for line in list(d)[2:16]:
            print("   ", line[:120])
    if absent:
        print(f"\nnot in the manuscript tree  : {absent}")
else:
    print(f"\nREFERENCE not found ({REFERENCE}); comparison skipped.")
    print("Upload the manuscript tables/ folder to Drive and set REFERENCE "
          "to it to compare.")
